# Lab 04.3 — HSV and Color Masks

## Objectives

- understand why HSV is useful for color segmentation;
- create binary masks with `cv2.inRange()`;
- compare masks for several colors;
- clean a mask using morphological operations.

The experiment first uses the controlled image from Lab 04.1 and then a webcam frame.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

image = cv2.imread("images/colored_shapes.png")
if image is None:
    raise FileNotFoundError("Run Lab04.1 once to generate images/colored_shapes.png")

hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

## 1. HSV thresholds

For an 8-bit OpenCV image, **H** (hue) ranges from 0 to 179; **S** (saturation) and **V** (brightness) range from 0 to 255. Hue describes the color, saturation distinguishes vivid colors from gray, and V describes brightness. Hue is unreliable on almost gray or very dark pixels, so we also set lower S and V limits.

Red wraps around the ends of the H interval and usually needs two ranges. `inRange()` returns a single-channel mask: **255 = selected pixel, 0 = rejected pixel**. These thresholds are starting points to adjust under your room lighting. HSV does not remove the effect of lighting.

In [ ]:
ranges = {
    "blue":   ((100, 100, 80), (130, 255, 255)),
    "green":  ((35, 80, 80),   (85, 255, 255)),
    "yellow": ((20, 100, 100), (34, 255, 255))
}

masks = {}

for name, (lower, upper) in ranges.items():
    masks[name] = cv2.inRange(hsv, np.array(lower), np.array(upper))

red1 = cv2.inRange(hsv, np.array((0, 100, 100)), np.array((10, 255, 255)))
red2 = cv2.inRange(hsv, np.array((170, 100, 100)), np.array((179, 255, 255)))
masks["red"] = cv2.bitwise_or(red1, red2)

In [ ]:
for name, mask in masks.items():
    plt.figure(figsize=(8, 3))
    plt.imshow(mask, cmap="gray")
    plt.title(name.upper() + " mask")
    plt.axis("off");

## 2. Apply a mask to the original image

In [ ]:
green_result = cv2.bitwise_and(image, image, mask=masks["green"])
green_rgb = cv2.cvtColor(green_result, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8, 4))
plt.imshow(green_rgb)
plt.title("Pixels selected by the green mask")
plt.axis("off");

## 3. Morphological cleanup

Real webcam images contain noise. Opening removes small isolated regions; closing can fill small holes.

In [ ]:
kernel = np.ones((5, 5), np.uint8)

clean = cv2.morphologyEx(masks["green"], cv2.MORPH_OPEN, kernel)
clean = cv2.morphologyEx(clean, cv2.MORPH_CLOSE, kernel)

plt.figure(figsize=(8, 3))
plt.imshow(clean, cmap="gray")
plt.title("Cleaned mask")
plt.axis("off");

## 4. Compare masks on your own camera image

Place a green object against a simple background. Run the supplied helper, then capture a frame. The camera is released before plotting. The clean synthetic image above needs little cleanup; this real image is better for comparing noise removal.

## Camera setup — provided code

Run this helper cell once in this notebook. It uses the previously verified **V4L2 / `/dev/video0` → 20 warm-up frames → capture → inline JPEG display → release** sequence. The requested resolution is 640×480; use `frame.shape` to check the actual resolution.

The `with camera_session()` block releases the camera through `finally`, including on Python errors or a normal Jupyter interrupt. Only one notebook/kernel should use the camera at a time. If Linux assigns a different capture device, change **CAMERA_DEVICE in each notebook you use**; notebook variables are not shared.

The helper is supplied; you do not need to implement it. Time limits are checked between reads: a USB driver call that blocks can delay an interrupt or timeout.

In [ ]:
import cv2
import numpy as np
import time
from contextlib import contextmanager
from IPython.display import display, Image, clear_output

CAMERA_DEVICE = "/dev/video0"
WARMUP_FRAMES = 20

def read_frame(cap):
    ok, frame = cap.read()
    if not ok or frame is None or frame.size == 0:
        raise RuntimeError(
            "No camera frame. Check the USB connection and close other camera notebooks."
        )
    return frame

@contextmanager
def camera_session():
    """Provided helper: open, warm up, and always release the camera."""
    cap = cv2.VideoCapture(CAMERA_DEVICE, cv2.CAP_V4L2)
    try:
        if not cap.isOpened():
            raise RuntimeError(
                "Cannot open " + CAMERA_DEVICE +
                ". Check the device and shut down other kernels using the camera."
            )
        # Request a modest resolution; the driver may negotiate another one.
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        warmup_start = time.monotonic()
        for _ in range(WARMUP_FRAMES):
            read_frame(cap)  # Discard frames while exposure settles.
            if time.monotonic() - warmup_start > 10:
                raise RuntimeError("Camera warm-up is too slow. Check the USB connection.")
        yield cap
    finally:
        cap.release()
        print("Camera released.")

def show_bgr(frame, replace=False):
    ok, jpg = cv2.imencode(".jpg", frame)
    if not ok:
        raise RuntimeError("JPEG encoding failed.")
    if replace:
        clear_output(wait=True)
    display(Image(data=jpg.tobytes()))


In [ ]:
with camera_session() as cap:
    frame = read_frame(cap)

print("Actual frame shape:", frame.shape)
hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
LOWER = (35, 80, 50)
UPPER = (85, 255, 255)
raw_mask = cv2.inRange(hsv_frame, np.array(LOWER), np.array(UPPER))
kernel = np.ones((5, 5), np.uint8)
clean_mask = cv2.morphologyEx(raw_mask, cv2.MORPH_OPEN, kernel)
clean_mask = cv2.morphologyEx(clean_mask, cv2.MORPH_CLOSE, kernel)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
axes[0].set_title("Camera image")
axes[1].imshow(raw_mask, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Raw mask")
axes[2].imshow(clean_mask, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("After OPEN + CLOSE")
for ax in axes:
    ax.axis("off")
plt.show()
plt.close(fig)
print("Selected pixels, raw / clean:",
      cv2.countNonZero(raw_mask), cv2.countNonZero(clean_mask))


## Short exercise

1. Observe which parts of your green object appear white in the mask.
2. Change only the minimum S value from 80 to 120 and compare the result.
3. Shade the object slightly and capture again. Explain one change you observe.

Opening can remove small objects as well as noise; closing can join nearby regions. A larger kernel is not always better. For another single-range color, use its limits from the table in the code above. Red requires the two-mask combination already demonstrated.